In [ ]:
from IPython.display import clear_output
clear_output(wait=True)

# End-to-End Three-Class Image Classification Pipeline

## Dataset
**Corn 3 Classes Image Dataset — Kaggle**

This notebook demonstrates a complete computer-vision workflow:

1. Install dependencies and configure reproducibility
2. Download a small, unusual three-class image dataset from Kaggle
3. Inspect and clean the extracted files
4. Perform exploratory data analysis
5. Build stratified train, validation, and test splits
6. Train a baseline CNN from scratch
7. Train a stronger lightweight MobileNetV2 transfer-learning model
8. Compare model metrics
9. Evaluate the selected model using a confusion matrix and classification report
10. Save the trained model and class metadata
11. Run single-image and batch inference

> The notebook is designed for Google Colab, Kaggle Notebooks, or a local Jupyter environment with Python 3.10+.

## 1. Install dependencies

Restart the runtime only if your environment requests it after installation.

In [ ]:
%pip install -q kaggle tensorflow scikit-learn pandas matplotlib pillow seaborn

#Connect to drive

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount("/content/drive")

# Set your working directory
WORK_DIR = "/content/drive/MyDrive/MLOps"

# Create it if it does not exist
os.makedirs(WORK_DIR, exist_ok=True)

# Change current working directory
os.chdir(WORK_DIR)

print("Current working directory:", os.getcwd())

## 2. Imports and configuration

In [ ]:
import json
import os
import random
import shutil
import zipfile
from collections import Counter
from pathlib import Path

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:


import tensorflow as tf

from PIL import Image, UnidentifiedImageError
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight



In [ ]:
print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))



In [ ]:
SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
BASELINE_EPOCHS = 15
TRANSFER_EPOCHS = 15
FINE_TUNE_EPOCHS = 8


In [ ]:
# Set this to an integer, such as 500, to deliberately keep only a
# small number of images per class for a fast classroom demonstration.
# Keep it as None to use all discovered images.
MAX_IMAGES_PER_CLASS = 500

DATASET_SLUG = "muratkokludataset/corn-3-classes-image-dataset"

ROOT_DIR = Path.cwd()
DOWNLOAD_DIR = ROOT_DIR / "kaggle_download"
EXTRACT_DIR = ROOT_DIR / "corn_dataset"
ARTIFACT_DIR = ROOT_DIR / "artifacts"



In [ ]:
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 3. Configure Kaggle credentials

Choose **one** method.

### Google Colab
1. Open Kaggle → Account Settings.
2. Create a legacy API token and download `kaggle.json`.
3. Run the upload cell below.

### Local Jupyter
Place `kaggle.json` at:

- Linux/macOS: `~/.kaggle/kaggle.json`
- Windows: `C:\Users\<username>\.kaggle\kaggle.json`

Do not publish your API token in Git.

In [ ]:
# COLAB ONLY: uncomment these lines to upload kaggle.json.
#
# from google.colab import files
# uploaded = files.upload()
#
# kaggle_dir = Path.home() / ".kaggle"
# kaggle_dir.mkdir(parents=True, exist_ok=True)
# shutil.copy("kaggle.json", kaggle_dir / "kaggle.json")
# os.chmod(kaggle_dir / "kaggle.json", 0o600)

## 4. Download and extract the dataset

The command is skipped when an archive is already present.

In [ ]:
archive_path = DOWNLOAD_DIR / "corn-3-classes-image-dataset.zip"

if not archive_path.exists():
    command = (
        f'kaggle datasets download -d "{DATASET_SLUG}" '
        f'-p "{DOWNLOAD_DIR}" --force'
    )
    exit_code = os.system(command)
    if exit_code != 0:
        raise RuntimeError(
            "Kaggle download failed. Verify kaggle.json, file permissions, "
            "internet connectivity, and dataset access."
        )
else:
    print("Archive already exists:", archive_path)

if not any(EXTRACT_DIR.iterdir()):
    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print("Extracted to:", EXTRACT_DIR)
else:
    print("Extraction directory is not empty; extraction skipped.")

print("\nTop-level extracted items:")
for path in sorted(EXTRACT_DIR.iterdir()):
    print(" -", path.name)

## 5. Discover image files and infer class labels

The notebook does not assume a fixed internal archive layout. It recursively finds image files and uses each image's immediate parent directory as its class label.

In [ ]:
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}



In [ ]:
all_image_paths = sorted(
    path
    for path in EXTRACT_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
)

if not all_image_paths:
    raise FileNotFoundError(
        f"No supported images were found under {EXTRACT_DIR.resolve()}."
    )



In [ ]:
raw_df = pd.DataFrame({
    "filepath": [str(path.resolve()) for path in all_image_paths],
    "label": [path.parent.name.strip() for path in all_image_paths],
})

print("Discovered image count:", len(raw_df))
display(raw_df.head())


In [ ]:
raw_counts = raw_df["label"].value_counts()
print("Raw class folders discovered:")
display(raw_counts.rename("image_count").to_frame())



In [ ]:
# Remove folders containing only a negligible number of images. Such folders
# are often thumbnails, metadata assets, or accidental extraction artifacts.
valid_labels = raw_counts[raw_counts >= 10].index.tolist()
df = raw_df[raw_df["label"].isin(valid_labels)].copy()

if df["label"].nunique() != 3:
    print(
        f"Expected exactly 3 usable classes, but found {df['label'].nunique()}:",
        sorted(df["label"].unique()),
    )
    print(
        "If the archive layout has changed, inspect raw_counts and explicitly "
        "set the three desired folder names below."
    )
    # Example:
    # desired_labels = ["Class_A", "Class_B", "Class_C"]
    # df = raw_df[raw_df["label"].isin(desired_labels)].copy()

assert df["label"].nunique() == 3, (
    "The dataset discovery step did not resolve exactly three classes."
)

print("\nResolved classes:", sorted(df["label"].unique()))
print("Resolved image count:", len(df))

## 6. Validate images and collect image metadata

Corrupted or unreadable files are excluded before splitting.

In [ ]:
records = []
bad_files = []

for filepath, label in df[["filepath", "label"]].itertuples(index=False):
    try:
        with Image.open(filepath) as image:
            image.verify()
            width, height = image.size
            mode = image.mode
        #print(records)
        records.append({
            "filepath": filepath,
            "label": label,
            "width": width,
            "height": height,
            "mode": mode,
            "aspect_ratio": width / max(height, 1),
        })
    except (UnidentifiedImageError, OSError, ValueError) as exc:
        bad_files.append((filepath, str(exc)))


In [ ]:
print("one data in records",records[0])

In [ ]:
image_df = pd.DataFrame(records)

print("Valid images:", len(image_df))
print("Unreadable images:", len(bad_files))

if bad_files:
    display(pd.DataFrame(bad_files, columns=["filepath", "error"]).head(10))

assert not image_df.empty, "No valid images remain after validation."
assert image_df["label"].nunique() == 3

In [ ]:
image_df.head()

## 7. Optionally create a smaller balanced working subset

This keeps the experiment fast and prevents a large class from dominating simply because it contains more files.

In [ ]:
if MAX_IMAGES_PER_CLASS is not None:
    sampled_parts = []
    for label, group in image_df.groupby("label"):
        n = min(MAX_IMAGES_PER_CLASS, len(group))
        sampled_parts.append(group.sample(n=n, random_state=SEED))
    image_df = (
        pd.concat(sampled_parts, ignore_index=True)
        .sample(frac=1, random_state=SEED)
        .reset_index(drop=True)
    )

print("Working dataset size:", len(image_df))


In [ ]:
display(image_df["label"].value_counts().rename("image_count").to_frame())

## 8. Exploratory data analysis

We inspect:

- Class distribution
- Image width and height
- Aspect-ratio variation
- Colour modes
- Random examples from each class

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(
    data=image_df,
    x="label",
    order=image_df["label"].value_counts().index,
    ax=axes[0],
)
axes[0].set_title("Class distribution")
axes[0].tick_params(axis="x", rotation=20)

sns.histplot(image_df["width"], bins=30, ax=axes[1])
axes[1].set_title("Image width distribution")

sns.histplot(image_df["height"], bins=30, ax=axes[2])
axes[2].set_title("Image height distribution")

plt.tight_layout()
plt.show()



In [ ]:
display(
    image_df.groupby("label")[["width", "height", "aspect_ratio"]]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

display(image_df["mode"].value_counts().rename("count").to_frame())

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# @title 🌽 Specific Image Inspector
IMAGE_INDEX = 60 # @param {type:"integer"}

# Ensure index is within bounds
if 0 <= IMAGE_INDEX < len(image_df):
    row = image_df.iloc[IMAGE_INDEX]

    plt.figure(figsize=(6, 6))
    with Image.open(row['filepath']) as img:
        plt.imshow(img.convert("RGB"))

    plt.title(f"Index: {IMAGE_INDEX} | Label: {row['label']}\nSize: {row['width']}x{row['height']}")
    plt.axis('off')
    plt.show()
else:
    print(f"Error: Index {IMAGE_INDEX} is out of range. Please enter a value between 0 and {len(image_df) - 1}.")

In [ ]:
class_names = sorted(image_df["label"].unique())
samples_per_class = 5

fig, axes = plt.subplots(
    len(class_names),
    samples_per_class,
    figsize=(15, 3.5 * len(class_names)),
)

for row_idx, label in enumerate(class_names):
    sample_rows = image_df[image_df["label"] == label].sample(
        n=min(samples_per_class, (image_df["label"] == label).sum()),
        random_state=SEED,
    )

    for col_idx, (_, row) in enumerate(sample_rows.iterrows()):
        with Image.open(row["filepath"]) as image:
            axes[row_idx, col_idx].imshow(image.convert("RGB"))
        axes[row_idx, col_idx].set_title(label)
        axes[row_idx, col_idx].axis("off")

plt.tight_layout()
plt.show()

## 9. Create stratified train, validation, and test splits

Target proportions:

- Training: 70%
- Validation: 15%
- Test: 15%

The test split remains untouched during training.

In [ ]:
train_df, temp_df = train_test_split(
    image_df,
    test_size=0.30,
    stratify=image_df["label"],
    random_state=SEED,
)



In [ ]:
train_df.describe()

In [ ]:
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED,
)



In [ ]:
val_df.describe()

In [ ]:
test_df.describe()

In [ ]:
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)


In [ ]:

split_summary = pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "validation": val_df["label"].value_counts(),
    "test": test_df["label"].value_counts(),
}).fillna(0).astype(int)



In [ ]:
display(split_summary)
print("Split sizes:", len(train_df), len(val_df), len(test_df))

## 10. Build TensorFlow input pipelines

The label order is fixed and reused for training, saving, and inference.

In [ ]:
class_names = sorted(train_df["label"].unique())
num_classes = len(class_names)
class_to_index = {name: idx for idx, name in enumerate(class_names)}

print("Class order:", class_names)



In [ ]:
def decode_and_resize(filepath, label_index):
    image_bytes = tf.io.read_file(filepath)
    image = tf.io.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False,
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)

    label = tf.one_hot(label_index, depth=num_classes)
    return image, label




In [ ]:
def make_dataset(frame, training=False):
    filepaths = frame["filepath"].astype(str).to_numpy()
    label_indices = (
        frame["label"]
        .map(class_to_index)
        .astype("int32")
        .to_numpy()
    )

    ds = tf.data.Dataset.from_tensor_slices(
        (filepaths, label_indices)
    )

    if training:
        ds = ds.shuffle(
            buffer_size=len(frame),
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    ds = ds.map(
        decode_and_resize,
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds



In [ ]:
train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)

images, labels = next(iter(train_ds))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

In [ ]:
images[0]

## 11. Compute class weights

Class weights reduce the effect of moderate class imbalance. On a perfectly balanced subset, the values will be close to 1.

In [ ]:
train_label_indices = train_df["label"].map(class_to_index).to_numpy()

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=train_label_indices,
)

class_weights = {idx: float(weight) for idx, weight in enumerate(weights)}
print("Class weights:", class_weights)

## 12. Common augmentation layer

In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.08),
        tf.keras.layers.RandomZoom(0.12),
        tf.keras.layers.RandomContrast(0.10),
    ],
    name="data_augmentation",
)

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
import numpy as np

# @title 🪄 Augmentation Visualizer
IMAGE_INDEX = 3 # @param {type:"integer"}

if 0 <= IMAGE_INDEX < len(image_df):
    # Load the specific image
    row = image_df.iloc[IMAGE_INDEX]
    img = Image.open(row['filepath']).convert("RGB")
    img_array = np.array(img.resize(IMG_SIZE))

    # Prepare for augmentation layer (needs batch dimension and float)
    img_tensor = tf.cast(tf.expand_dims(img_array, 0), tf.float32)

    plt.figure(figsize=(15, 8))

    # Show Original
    plt.subplot(2, 3, 1)
    plt.imshow(img_array.astype("uint8"))
    plt.title(f"Original (Index {IMAGE_INDEX})\n{row['label']}")
    plt.axis("off")

    # Show 5 random augmentations
    for i in range(5):
        augmented_img = data_augmentation(img_tensor, training=True)
        plt.subplot(2, 3, i + 2)
        plt.imshow(augmented_img[0].numpy().astype("uint8"))
        plt.title(f"Augmented Version {i+1}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print(f"Error: Index {IMAGE_INDEX} is out of range (0 to {len(image_df)-1})")

# Model 1 — Baseline CNN trained from scratch

This deliberately simple network establishes a baseline. It learns every visual feature only from this dataset.

https://towardsdatascience.com/computer-vision-convolution-basics-2d0ae3b79346/

In [ ]:
tf.keras.backend.clear_session()

baseline_model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(*IMG_SIZE, 3)),
        data_augmentation,
        tf.keras.layers.Rescaling(1.0 / 255),

        tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.35),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ],
    name="baseline_cnn",
)

baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top_2_accuracy"),
    ],
)

baseline_model.summary()

In [ ]:
baseline_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        ARTIFACT_DIR / "baseline_best.keras",
        monitor="val_loss",
        save_best_only=True,
    ),
]

baseline_history = baseline_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=BASELINE_EPOCHS,
    class_weight=class_weights,
    callbacks=baseline_callbacks,
)

## 13. Plot baseline learning curves

In [ ]:
def plot_history(history, title):
    history_df = pd.DataFrame(history.history)
    epochs = np.arange(1, len(history_df) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs, history_df["loss"], label="train")
    axes[0].plot(epochs, history_df["val_loss"], label="validation")
    axes[0].set_title(f"{title}: loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, history_df["accuracy"], label="train")
    axes[1].plot(epochs, history_df["val_accuracy"], label="validation")
    axes[1].set_title(f"{title}: accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_history(baseline_history, "Baseline CNN")

In [ ]:
baseline_test_metrics = baseline_model.evaluate(
    test_ds,
    return_dict=True,
    verbose=1,
)
baseline_test_metrics

In [ ]:
def collect_predictions(model, dataset):
    y_true = []
    probabilities = []

    for image_batch, label_batch in dataset:
        batch_probabilities = model.predict(image_batch, verbose=0)
        probabilities.append(batch_probabilities)
        y_true.extend(np.argmax(label_batch.numpy(), axis=1))

    probabilities = np.concatenate(probabilities, axis=0)
    y_pred = np.argmax(probabilities, axis=1)

    return np.asarray(y_true), y_pred, probabilities

y_true, y_pred, y_prob = collect_predictions(baseline_model, test_ds)

report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report).transpose()
display(report_df.round(4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.title("Confusion matrix")
plt.tight_layout()
plt.show()

In [ ]:
test_result_df = test_df.copy()
test_result_df["true_index"] = y_true
test_result_df["pred_index"] = y_pred
test_result_df["predicted_label"] = [
    class_names[index] for index in y_pred
]
test_result_df["confidence"] = y_prob.max(axis=1)
test_result_df["correct"] = (
    test_result_df["true_index"] == test_result_df["pred_index"]
)

mistakes = (
    test_result_df[~test_result_df["correct"]]
    .sort_values("confidence", ascending=False)
    .head(12)
)

if mistakes.empty:
    print("No misclassified images were found in this test split.")
else:
    cols = 4
    rows = int(np.ceil(len(mistakes) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes = np.asarray(axes).reshape(-1)

    for axis in axes:
        axis.axis("off")

    for axis, (_, row) in zip(axes, mistakes.iterrows()):
        with Image.open(row["filepath"]) as image:
            axis.imshow(image.convert("RGB"))
        axis.set_title(
            f"True: {row['label']}\n"
            f"Pred: {row['predicted_label']} "
            f"({row['confidence']:.1%})"
        )
        axis.axis("off")

    plt.tight_layout()
    plt.show()

# Model 2 — MobileNetV2 transfer learning

MobileNetV2 is lightweight and commonly used where inference speed and model size matter. Its pretrained visual features should be more data-efficient than learning from scratch.

Training occurs in two stages:

1. Freeze the pretrained backbone and train only the new classifier head.
2. Unfreeze the upper backbone layers and fine-tune them using a much smaller learning rate.

In [ ]:
tf.keras.backend.clear_session()

augmentation_transfer = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.08),
        tf.keras.layers.RandomZoom(0.12),
        tf.keras.layers.RandomContrast(0.10),
    ],
    name="transfer_augmentation",
)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = augmentation_transfer(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.30)(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.20)(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

transfer_model = tf.keras.Model(inputs, outputs, name="mobilenetv2_classifier")

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top_2_accuracy"),
    ],
)

transfer_model.summary()

In [ ]:
transfer_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-6,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        ARTIFACT_DIR / "mobilenetv2_head_best.keras",
        monitor="val_loss",
        save_best_only=True,
    ),
]

transfer_history = transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=TRANSFER_EPOCHS,
    class_weight=class_weights,
    callbacks=transfer_callbacks,
)

In [ ]:
plot_history(transfer_history, "MobileNetV2 frozen backbone")

## 14. Fine-tune the upper MobileNetV2 layers

In [ ]:
base_model.trainable = True

# Keep most pretrained layers frozen. Unfreeze only the upper portion.
fine_tune_at = max(0, len(base_model.layers) - 30)

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Batch-normalisation layers can become unstable on small datasets.
for layer in base_model.layers[fine_tune_at:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top_2_accuracy"),
    ],
)

fine_tune_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        ARTIFACT_DIR / "mobilenetv2_finetuned_best.keras",
        monitor="val_loss",
        save_best_only=True,
    ),
]

fine_tune_history = transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    class_weight=class_weights,
    callbacks=fine_tune_callbacks,
)

In [ ]:
plot_history(fine_tune_history, "MobileNetV2 fine-tuning")

In [ ]:
transfer_test_metrics = transfer_model.evaluate(
    test_ds,
    return_dict=True,
    verbose=1,
)
transfer_test_metrics

## 15. Compare both models

Select the winner using validation/test performance, not training accuracy.

In [ ]:
comparison_df = pd.DataFrame(
    [
        {"model": "Baseline CNN", **baseline_test_metrics},
        {"model": "MobileNetV2", **transfer_test_metrics},
    ]
).set_index("model")

display(comparison_df.round(4))

best_model = (
    transfer_model
    if transfer_test_metrics["accuracy"] >= baseline_test_metrics["accuracy"]
    else baseline_model
)
best_model_name = (
    "MobileNetV2"
    if best_model is transfer_model
    else "Baseline CNN"
)

print("Selected model:", best_model_name)

## 16. Detailed evaluation

Metrics include:

- Accuracy
- Per-class precision
- Per-class recall
- Per-class F1-score
- Confusion matrix

A model can have acceptable overall accuracy while performing poorly on one class, so the per-class results matter.

In [ ]:
def collect_predictions(model, dataset):
    y_true = []
    probabilities = []

    for image_batch, label_batch in dataset:
        batch_probabilities = model.predict(image_batch, verbose=0)
        probabilities.append(batch_probabilities)
        y_true.extend(np.argmax(label_batch.numpy(), axis=1))

    probabilities = np.concatenate(probabilities, axis=0)
    y_pred = np.argmax(probabilities, axis=1)

    return np.asarray(y_true), y_pred, probabilities

y_true, y_pred, y_prob = collect_predictions(best_model, test_ds)

report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report).transpose()
display(report_df.round(4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.title(f"Confusion matrix — {best_model_name}")
plt.tight_layout()
plt.show()

## 17. Inspect misclassified images

In [ ]:
test_result_df = test_df.copy()
test_result_df["true_index"] = y_true
test_result_df["pred_index"] = y_pred
test_result_df["predicted_label"] = [
    class_names[index] for index in y_pred
]
test_result_df["confidence"] = y_prob.max(axis=1)
test_result_df["correct"] = (
    test_result_df["true_index"] == test_result_df["pred_index"]
)

mistakes = (
    test_result_df[~test_result_df["correct"]]
    .sort_values("confidence", ascending=False)
    .head(12)
)

if mistakes.empty:
    print("No misclassified images were found in this test split.")
else:
    cols = 4
    rows = int(np.ceil(len(mistakes) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes = np.asarray(axes).reshape(-1)

    for axis in axes:
        axis.axis("off")

    for axis, (_, row) in zip(axes, mistakes.iterrows()):
        with Image.open(row["filepath"]) as image:
            axis.imshow(image.convert("RGB"))
        axis.set_title(
            f"True: {row['label']}\n"
            f"Pred: {row['predicted_label']} "
            f"({row['confidence']:.1%})"
        )
        axis.axis("off")

    plt.tight_layout()
    plt.show()

## 18. Save model and metadata

In [ ]:
final_model_path = ARTIFACT_DIR / "corn_classifier.keras"
metadata_path = ARTIFACT_DIR / "class_names.json"
metrics_path = ARTIFACT_DIR / "model_comparison.csv"

best_model.save(final_model_path)

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(
        {
            "class_names": class_names,
            "image_size": list(IMG_SIZE),
            "selected_model": best_model_name,
        },
        file,
        indent=2,
    )

comparison_df.to_csv(metrics_path)

print("Saved model:", final_model_path.resolve())
print("Saved metadata:", metadata_path.resolve())
print("Saved comparison:", metrics_path.resolve())

# Inference pipeline

The following functions:

1. Load the saved model and metadata
2. Validate and preprocess an input image
3. Return the top prediction and all class probabilities
4. Support either one image or an entire directory

In [ ]:
def load_classifier(
    model_path=ARTIFACT_DIR / "corn_classifier.keras",
    metadata_path=ARTIFACT_DIR / "class_names.json",
):
    model = tf.keras.models.load_model(model_path)

    with open(metadata_path, "r", encoding="utf-8") as file:
        metadata = json.load(file)

    return model, metadata


def prepare_image(image_path, image_size):
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image does not exist: {image_path}")

    try:
        image = Image.open(image_path).convert("RGB")
    except (UnidentifiedImageError, OSError) as exc:
        raise ValueError(f"Unable to read image: {image_path}") from exc

    original_image = image.copy()
    resized = image.resize(tuple(image_size))
    array = np.asarray(resized, dtype=np.float32)
    batch = np.expand_dims(array, axis=0)

    return original_image, batch


def predict_image(
    image_path,
    model,
    metadata,
    show=True,
):
    original_image, batch = prepare_image(
        image_path,
        metadata["image_size"],
    )

    probabilities = model.predict(batch, verbose=0)[0]
    class_names_local = metadata["class_names"]
    predicted_index = int(np.argmax(probabilities))

    result = {
        "image_path": str(Path(image_path).resolve()),
        "predicted_class": class_names_local[predicted_index],
        "confidence": float(probabilities[predicted_index]),
        "probabilities": {
            class_name: float(probability)
            for class_name, probability in zip(
                class_names_local,
                probabilities,
            )
        },
    }

    if show:
        plt.figure(figsize=(6, 5))
        plt.imshow(original_image)
        plt.axis("off")
        plt.title(
            f"{result['predicted_class']} "
            f"({result['confidence']:.2%})"
        )
        plt.show()

        display(
            pd.DataFrame(
                result["probabilities"].items(),
                columns=["class", "probability"],
            ).sort_values("probability", ascending=False)
        )

    return result


def predict_directory(
    directory,
    model,
    metadata,
):
    directory = Path(directory)

    if not directory.exists():
        raise FileNotFoundError(f"Directory does not exist: {directory}")

    image_paths = sorted(
        path
        for path in directory.rglob("*")
        if path.is_file()
        and path.suffix.lower() in SUPPORTED_EXTENSIONS
    )

    results = []

    for image_path in image_paths:
        try:
            result = predict_image(
                image_path,
                model,
                metadata,
                show=False,
            )
            results.append({
                "filepath": result["image_path"],
                "predicted_class": result["predicted_class"],
                "confidence": result["confidence"],
            })
        except Exception as exc:
            results.append({
                "filepath": str(image_path.resolve()),
                "predicted_class": None,
                "confidence": None,
                "error": str(exc),
            })

    return pd.DataFrame(results)

## 19. Single-image inference example

In [ ]:
inference_model, inference_metadata = load_classifier()

example_image = test_df.sample(1, random_state=7).iloc[0]["filepath"]
single_result = predict_image(
    example_image,
    inference_model,
    inference_metadata,
)
single_result

## 20. Batch inference example

In [ ]:
# Replace this directory with a folder containing new images.
example_directory = Path(test_df.iloc[0]["filepath"]).parent

batch_predictions = predict_directory(
    example_directory,
    inference_model,
    inference_metadata,
)

display(batch_predictions.head(20))

## 21. Optional TensorFlow Lite export

This produces a smaller deployment artifact suitable for mobile, edge, or lightweight serving environments.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)

# Dynamic-range quantisation reduces model size without requiring
# a representative calibration dataset.
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()
tflite_path = ARTIFACT_DIR / "corn_classifier_dynamic_quant.tflite"
tflite_path.write_bytes(tflite_model)

print("Saved TensorFlow Lite model:", tflite_path.resolve())
print("Size in MB:", round(tflite_path.stat().st_size / (1024 ** 2), 2))

In [ ]:
import shutil
from google.colab import files

# Define the zip filename
zip_filename = "corn_classification_artifacts.zip"

# Create the zip archive
shutil.make_archive("corn_classification_artifacts", 'zip', ARTIFACT_DIR)

print(f"Successfully created {zip_filename}")

# Trigger download
files.download(zip_filename)

In [ ]:
import shutil
from google.colab import files

# Define the zip filename
zip_filename = "corn_classification_artifacts_data.zip"

# Create the zip archive
shutil.make_archive("corn_classification_artifacts_data", 'zip',EXTRACT_DIR )

print(f"Successfully created {zip_filename}")

# Trigger download
files.download(zip_filename)

# Final interpretation checklist

After running the notebook, discuss these points:

1. Was the dataset balanced?
2. Did image dimensions or aspect ratios vary significantly?
3. Did the scratch CNN overfit?
4. How much did transfer learning improve test accuracy and macro F1?
5. Which classes were confused most often?
6. Were high-confidence mistakes caused by poor labels, difficult images, background bias, or class similarity?
7. Is the final model accurate enough for a demonstration, or would production use require more independently collected data?
8. Does the saved inference pipeline reproduce the same class order used during training?

## Important limitation

A high score on a small random split does not prove production readiness. Near-duplicate images, consistent backgrounds, or images captured under the same conditions can inflate test performance. A serious deployment should be validated against a separate real-world dataset collected from different farms, cameras, locations, and lighting conditions.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# @title 🌽 Specific Image Inspector
IMAGE_INDEX = 60 # @param {type:"integer"}

# Ensure index is within bounds
if 0 <= IMAGE_INDEX < len(image_df):
    row = image_df.iloc[IMAGE_INDEX]

    plt.figure(figsize=(6, 6))
    with Image.open(row['filepath']) as img:
        plt.imshow(img.convert("RGB"))

    plt.title(f"Index: {IMAGE_INDEX} | Label: {row['label']}\nSize: {row['width']}x{row['height']}")
    plt.axis('off')
    plt.show()
else:
    print(f"Error: Index {IMAGE_INDEX} is out of range. Please enter a value between 0 and {len(image_df) - 1}.")